# Notebook 05 — Feature Engineering (v1)

**Objectif** : Créer les variables dérivées validées par l'analyse statistique (notebook 04) pour alimenter la modélisation.

**Scope v1** : Variables calculables avec les données football-data.co.uk uniquement (1993-2025, 12 614 matchs).

**Variables créées (8 par équipe)** :
1. **Form_Last5** : Points cumulés sur 5 derniers matchs (0-15) — validé η²=0.0160
2. **Rank** : Classement avant le match (1-20+) — validé η²=0.0300
3. **Streak** : Série victoires/défaites consécutives (+/- N) — validé η²=0.0083
4. **Goal_Diff_Cumul** : Différence de buts cumulée en cours de saison
5. **Rolling_Goals_For** : Moyenne mobile buts marqués (5 matchs)
6. **Rolling_Goals_Against** : Moyenne mobile buts encaissés (5 matchs)
7. **Points_Pace** : Projection points sur 38 matchs
8. **Match_Number** : Position dans la saison (1-38)

**Format de sortie** : 1 ligne = 1 match avec features Home/Away (12.6K lignes)

**Export** : `data/processed/pl_features_v1.csv`

**Principe critique** : `.shift(1)` systématique pour éviter data leakage (features calculées avec infos **avant** le match uniquement).

---

## 1. Chargement des données


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Détection racine projet
current_path = Path.cwd()
while not (current_path / 'data').exists() and current_path != current_path.parent:
    current_path = current_path.parent

PROJECT_ROOT = current_path
DATA_INTERIM = PROJECT_ROOT / 'data' / 'interim'
DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'

print(f"Racine projet : {PROJECT_ROOT}")
print(f"Chargement depuis : {DATA_INTERIM / 'pl_cleaned.csv'}")
print(f"Export vers : {DATA_PROCESSED}")

# Création dossier processed si inexistant
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)


Racine projet : /workspace
Chargement depuis : /workspace/data/interim/pl_cleaned.csv
Export vers : /workspace/data/processed


In [3]:
# Chargement des données nettoyées
df = pd.read_csv(DATA_INTERIM / 'pl_cleaned.csv')

# Conversion Date en datetime
df['Date'] = pd.to_datetime(df['Date'])

# Tri chronologique (important pour calculs temporels)
df = df.sort_values(['Season', 'Date']).reset_index(drop=True)

# Compte équipes uniques
all_teams = pd.concat([df['HomeTeam'], df['AwayTeam']]).unique()

print(f"Dataset chargé : {df.shape}")
print(f"Période : {df['Date'].min()} → {df['Date'].max()}")
print(f"Saisons : {df['Season'].nunique()}")
print(f"Équipes uniques : {len(all_teams)}")
print(f"\nColonnes disponibles : {list(df.columns)}")


Dataset chargé : (12614, 26)
Période : 1993-08-14 00:00:00 → 2026-05-24 00:00:00
Saisons : 33
Équipes uniques : 51

Colonnes disponibles : ['Date', 'Season', 'HomeTeam', 'AwayTeam', 'Referee', 'FTHG', 'FTAG', 'FTR', 'HTHG', 'HTAG', 'HTR', 'HS', 'AS', 'HST', 'AST', 'HC', 'AC', 'HF', 'AF', 'HY', 'AY', 'HR', 'AR', 'AvgH', 'AvgD', 'AvgA']


---

## 2. Expansion du dataset (format "par équipe")

Pour calculer les features temporelles (forme, classement, streak), on doit transformer le dataset :
- **Format actuel** : 1 ligne = 1 match (HomeTeam vs AwayTeam)
- **Format nécessaire** : 1 ligne = 1 équipe dans 1 match (pour calculs groupés par équipe)

Chaque match génère **2 lignes** :
- Ligne 1 : perspective HomeTeam (opponent = AwayTeam, isHome = True)
- Ligne 2 : perspective AwayTeam (opponent = HomeTeam, isHome = False)

Permet ensuite de grouper par `(Team, Season)` et calculer rolling stats avec `.shift(1)`.


In [4]:
# Expansion : 1 match → 2 lignes (Home + Away)
df_expanded = []

for _, row in df.iterrows():
    # Ligne 1 : perspective Home
    home_row = {
        'Date': row['Date'],
        'Season': row['Season'],
        'Team': row['HomeTeam'],
        'Opponent': row['AwayTeam'],
        'IsHome': True,
        'Goals_For': row['FTHG'],
        'Goals_Against': row['FTAG'],
        'Result': row['FTR'],  # H/D/A du point de vue du match original
        'Referee': row['Referee']
    }
    
    # Ligne 2 : perspective Away
    away_row = {
        'Date': row['Date'],
        'Season': row['Season'],
        'Team': row['AwayTeam'],
        'Opponent': row['HomeTeam'],
        'IsHome': False,
        'Goals_For': row['FTAG'],
        'Goals_Against': row['FTHG'],
        'Result': row['FTR'],  # Gardé tel quel pour l'instant
        'Referee': row['Referee']
    }
    
    df_expanded.append(home_row)
    df_expanded.append(away_row)

df_expanded = pd.DataFrame(df_expanded)

# Conversion Result du point de vue de l'équipe (W/D/L)
def convert_result_to_team_perspective(row):
    if row['IsHome']:
        # Home perspective : H=W, D=D, A=L
        return {'H': 'W', 'D': 'D', 'A': 'L'}[row['Result']]
    else:
        # Away perspective : H=L, D=D, A=W
        return {'H': 'L', 'D': 'D', 'A': 'W'}[row['Result']]

df_expanded['Result_Team'] = df_expanded.apply(convert_result_to_team_perspective, axis=1)

# Tri chronologique par équipe
df_expanded = df_expanded.sort_values(['Team', 'Season', 'Date']).reset_index(drop=True)

print(f"Dataset expandé : {df_expanded.shape}")
print(f"Équipes uniques : {df_expanded['Team'].nunique()}")
print(f"\n✓ Chaque match est maintenant représenté par 2 lignes (une par équipe)")
print(f"→ {len(df)} matchs × 2 = {len(df_expanded)} lignes")
print(f"\nAperçu :")
print(df_expanded.head(10))


Dataset expandé : (25228, 10)
Équipes uniques : 51

✓ Chaque match est maintenant représenté par 2 lignes (une par équipe)
→ 12614 matchs × 2 = 25228 lignes

Aperçu :
        Date  Season     Team     Opponent  IsHome  Goals_For  Goals_Against  \
0 2000-08-19       1  Arsenal   Sunderland   False        0.0            1.0   
1 2000-08-21       1  Arsenal    Liverpool    True        2.0            0.0   
2 2000-08-26       1  Arsenal     Charlton    True        5.0            3.0   
3 2000-09-06       1  Arsenal      Chelsea   False        2.0            2.0   
4 2000-09-09       1  Arsenal     Bradford   False        1.0            1.0   
5 2000-09-16       1  Arsenal     Coventry    True        2.0            1.0   
6 2000-09-23       1  Arsenal      Ipswich   False        1.0            1.0   
7 2000-10-01       1  Arsenal   Man United    True        1.0            0.0   
8 2000-10-14       1  Arsenal  Aston Villa    True        1.0            0.0   
9 2000-10-21       1  Arsenal    

---

## 3. Calcul des features temporelles (par équipe × saison)

**Principe critique** : `.shift(1)` systématique pour éviter **data leakage**.

Chaque feature doit être calculée avec les informations **disponibles avant le match** :
- Rolling stats : fenêtre glissante **avant** le match courant
- Classement : position **avant** le match courant
- Streak : série **avant** le match courant

**Variables créées** :
1. **Points_Match** : 3 (W), 1 (D), 0 (L)
2. **Form_Last5** : rolling sum points sur 5 matchs (shift 1)
3. **Rank** : classement avant le match (basé sur points cumulés shift 1)
4. **Streak** : série victoires/défaites consécutives (shift 1)
5. **Goal_Diff_Cumul** : différence buts cumulée (shift 1)
6. **Rolling_Goals_For** : moyenne mobile buts marqués 5 matchs (shift 1)
7. **Rolling_Goals_Against** : moyenne mobile buts encaissés 5 matchs (shift 1)
8. **Points_Pace** : projection points sur 38 matchs (shift 1)
9. **Match_Number** : numéro du match dans la saison (1-38)


In [5]:
# Calcul des points par match
def calculate_points(result):
    return {'W': 3, 'D': 1, 'L': 0}[result]

df_expanded['Points_Match'] = df_expanded['Result_Team'].apply(calculate_points)

# Groupement par équipe × saison pour calculs temporels
df_expanded = df_expanded.sort_values(['Team', 'Season', 'Date']).reset_index(drop=True)

# FEATURE 1 : Form_Last5 (rolling sum 5 matchs avec shift 1)
df_expanded['Form_Last5'] = (
    df_expanded.groupby(['Team', 'Season'])['Points_Match']
    .transform(lambda x: x.rolling(window=5, min_periods=1).sum().shift(1))
)

# FEATURE 9 : Match_Number (position dans la saison)
df_expanded['Match_Number'] = (
    df_expanded.groupby(['Team', 'Season']).cumcount() + 1
)

print("✓ Features créées : Points_Match, Form_Last5, Match_Number")
print(f"\nDistribution Form_Last5 (après shift, donc NaN pour 1er match de chaque saison) :")
print(df_expanded['Form_Last5'].describe())
print(f"\nAperçu Arsenal saison 2000 (10 premiers matchs) :")
print(df_expanded[df_expanded['Team'] == 'Arsenal'].head(10)[['Date', 'Opponent', 'Result_Team', 'Points_Match', 'Match_Number', 'Form_Last5']])


✓ Features créées : Points_Match, Form_Last5, Match_Number

Distribution Form_Last5 (après shift, donc NaN pour 1er match de chaque saison) :
count    24564.000000
mean         6.495115
std          3.460624
min          0.000000
25%          4.000000
50%          6.000000
75%          9.000000
max         15.000000
Name: Form_Last5, dtype: float64

Aperçu Arsenal saison 2000 (10 premiers matchs) :
        Date     Opponent Result_Team  Points_Match  Match_Number  Form_Last5
0 2000-08-19   Sunderland           L             0             1         NaN
1 2000-08-21    Liverpool           W             3             2         0.0
2 2000-08-26     Charlton           W             3             3         3.0
3 2000-09-06      Chelsea           D             1             4         6.0
4 2000-09-09     Bradford           D             1             5         7.0
5 2000-09-16     Coventry           W             3             6         8.0
6 2000-09-23      Ipswich           D             1 

In [10]:
# Rechargement depuis le début pour éviter problèmes de colonnes perdues
df = pd.read_csv(DATA_INTERIM / 'pl_cleaned.csv')
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values(['Season', 'Date']).reset_index(drop=True)

# Expansion : 1 match → 2 lignes
df_expanded = []
for _, row in df.iterrows():
    home_row = {
        'Date': row['Date'],
        'Season': row['Season'],
        'Team': row['HomeTeam'],
        'Opponent': row['AwayTeam'],
        'IsHome': 1,  # Entier au lieu de booléen
        'Goals_For': row['FTHG'],
        'Goals_Against': row['FTAG'],
        'Result': row['FTR'],
        'Referee': row['Referee']
    }
    away_row = {
        'Date': row['Date'],
        'Season': row['Season'],
        'Team': row['AwayTeam'],
        'Opponent': row['HomeTeam'],
        'IsHome': 0,
        'Goals_For': row['FTAG'],
        'Goals_Against': row['FTHG'],
        'Result': row['FTR'],
        'Referee': row['Referee']
    }
    df_expanded.append(home_row)
    df_expanded.append(away_row)

df_expanded = pd.DataFrame(df_expanded)

# Conversion Result du point de vue de l'équipe
def convert_result(row):
    if row['IsHome'] == 1:
        return {'H': 'W', 'D': 'D', 'A': 'L'}[row['Result']]
    else:
        return {'H': 'L', 'D': 'D', 'A': 'W'}[row['Result']]

df_expanded['Result_Team'] = df_expanded.apply(convert_result, axis=1)

# Tri par équipe, saison, date
df_expanded = df_expanded.sort_values(['Team', 'Season', 'Date']).reset_index(drop=True)

# Calcul Points
df_expanded['Points_Match'] = df_expanded['Result_Team'].map({'W': 3, 'D': 1, 'L': 0})

print("✓ Rechargement et expansion OK")
print(f"Shape : {df_expanded.shape}")
print(f"Colonnes : {df_expanded.columns.tolist()}")


✓ Rechargement et expansion OK
Shape : (25228, 11)
Colonnes : ['Date', 'Season', 'Team', 'Opponent', 'IsHome', 'Goals_For', 'Goals_Against', 'Result', 'Referee', 'Result_Team', 'Points_Match']


In [11]:
# FEATURE 1 : Form_Last5 (rolling sum 5 matchs avec shift 1)
df_expanded['Form_Last5'] = (
    df_expanded.groupby(['Team', 'Season'])['Points_Match']
    .transform(lambda x: x.rolling(window=5, min_periods=1).sum().shift(1))
)

# FEATURE 2 : Match_Number (position dans la saison)
df_expanded['Match_Number'] = (
    df_expanded.groupby(['Team', 'Season']).cumcount() + 1
)

# FEATURE 3 : Points_Cumul_Before (shift 1)
df_expanded['Points_Cumul'] = (
    df_expanded.groupby(['Team', 'Season'])['Points_Match']
    .transform(lambda x: x.cumsum().shift(1))
)

# FEATURE 4 : Goal_Diff_Cumul (shift 1)
df_expanded['Goal_Diff_Cumul'] = (
    df_expanded.groupby(['Team', 'Season'])['Goals_For']
    .transform(lambda x: x.cumsum().shift(1))
    - df_expanded.groupby(['Team', 'Season'])['Goals_Against']
    .transform(lambda x: x.cumsum().shift(1))
)

# FEATURE 5 : Rolling_Goals_For (moyenne mobile 5 matchs avec shift 1)
df_expanded['Rolling_Goals_For'] = (
    df_expanded.groupby(['Team', 'Season'])['Goals_For']
    .transform(lambda x: x.rolling(window=5, min_periods=1).mean().shift(1))
)

# FEATURE 6 : Rolling_Goals_Against (moyenne mobile 5 matchs avec shift 1)
df_expanded['Rolling_Goals_Against'] = (
    df_expanded.groupby(['Team', 'Season'])['Goals_Against']
    .transform(lambda x: x.rolling(window=5, min_periods=1).mean().shift(1))
)

# FEATURE 7 : Points_Pace (projection sur 38 matchs, shift 1)
df_expanded['Points_Pace'] = (
    df_expanded['Points_Cumul'] / (df_expanded['Match_Number'] - 1) * 38
).replace([np.inf, -np.inf], np.nan)  # Division par 0 pour match 1

# FEATURE 8 : Streak (série victoires/défaites consécutives avec shift 1)
def calculate_streak(group):
    """Calcule la série avant chaque match (shift 1)"""
    streaks = []
    current_streak = 0
    
    for result in group['Result_Team']:
        streaks.append(current_streak)  # Append avant mise à jour = shift automatique
        
        if result == 'W':
            current_streak = current_streak + 1 if current_streak >= 0 else 1
        elif result == 'L':
            current_streak = current_streak - 1 if current_streak <= 0 else -1
        else:  # Draw
            current_streak = 0
    
    return streaks

df_expanded['Streak'] = (
    df_expanded.groupby(['Team', 'Season'], group_keys=False)
    .apply(lambda g: pd.Series(calculate_streak(g), index=g.index))
)

print("✓ 8 features temporelles créées avec shift(1)")
print(f"\nNombre de lignes avec features : {len(df_expanded)}")
print(f"Nombre de NaN par feature (attendu pour 1er match de chaque saison) :")
print(df_expanded[['Form_Last5', 'Points_Cumul', 'Goal_Diff_Cumul', 
                   'Rolling_Goals_For', 'Rolling_Goals_Against', 
                   'Points_Pace', 'Streak']].isna().sum())


✓ 8 features temporelles créées avec shift(1)

Nombre de lignes avec features : 25228
Nombre de NaN par feature (attendu pour 1er match de chaque saison) :
Form_Last5               664
Points_Cumul             664
Goal_Diff_Cumul          664
Rolling_Goals_For        664
Rolling_Goals_Against    664
Points_Pace              664
Streak                     0
dtype: int64


In [12]:
# FEATURE 9 : Rank (classement avant le match)
# Pour chaque (Season, Date), on classe les équipes par Points_Cumul et Goal_Diff_Cumul

# Initialisation Rank à NaN
df_expanded['Rank'] = np.nan

# Calcul du classement pour chaque journée
for (season, date), group_idx in df_expanded.groupby(['Season', 'Date']).groups.items():
    # Extraction du groupe
    group = df_expanded.loc[group_idx].copy()
    
    # Tri par Points_Cumul (décroissant) puis Goal_Diff_Cumul (décroissant)
    # NaN en dernier (équipes au 1er match)
    group_sorted = group.sort_values(
        ['Points_Cumul', 'Goal_Diff_Cumul'], 
        ascending=[False, False],
        na_position='last'
    )
    
    # Attribution du rang
    ranks = list(range(1, len(group_sorted) + 1))
    
    # Assignation dans le dataframe original
    df_expanded.loc[group_sorted.index, 'Rank'] = ranks

print("✓ Feature Rank créée")
print(f"\nDistribution Rank :")
print(df_expanded['Rank'].describe())
print(f"\nNombre de NaN Rank : {df_expanded['Rank'].isna().sum()}")
print(f"\nVérification : exemple Arsenal saison 2000 (premiers matchs)")
print(df_expanded[df_expanded['Team'] == 'Arsenal'].head(10)[
    ['Date', 'Opponent', 'Match_Number', 'Result_Team', 'Points_Cumul', 
     'Goal_Diff_Cumul', 'Rank', 'Form_Last5', 'Streak']
])


✓ Feature Rank créée

Distribution Rank :
count    25228.000000
mean         6.255827
std          4.550248
min          1.000000
25%          2.000000
50%          5.000000
75%          9.000000
max         22.000000
Name: Rank, dtype: float64

Nombre de NaN Rank : 0

Vérification : exemple Arsenal saison 2000 (premiers matchs)
        Date     Opponent  Match_Number Result_Team  Points_Cumul  \
0 2000-08-19   Sunderland             1           L           NaN   
1 2000-08-21    Liverpool             2           W           0.0   
2 2000-08-26     Charlton             3           W           3.0   
3 2000-09-06      Chelsea             4           D           6.0   
4 2000-09-09     Bradford             5           D           7.0   
5 2000-09-16     Coventry             6           W           8.0   
6 2000-09-23      Ipswich             7           D          11.0   
7 2000-10-01   Man United             8           W          12.0   
8 2000-10-14  Aston Villa             9         

---

## 4. Reconstitution format "match-complete"

Actuellement : **1 ligne = 1 équipe dans 1 match** (25 228 lignes)  
Objectif : **1 ligne = 1 match avec features Home/Away** (12 614 lignes)

Format final pour modélisation :
- Colonnes Home : `Home_Form`, `Home_Rank`, `Home_Streak`, etc.
- Colonnes Away : `Away_Form`, `Away_Rank`, `Away_Streak`, etc.
- Target : `FTR` (H/D/A)

Ce format permet au modèle de voir **simultanément** les features des 2 équipes pour prédire le résultat.


In [13]:
# Séparation Home et Away
df_home = df_expanded[df_expanded['IsHome'] == 1].copy()
df_away = df_expanded[df_expanded['IsHome'] == 0].copy()

# Renommage colonnes Home
df_home = df_home.rename(columns={
    'Team': 'HomeTeam',
    'Opponent': 'AwayTeam',
    'Goals_For': 'FTHG',
    'Goals_Against': 'FTAG',
    'Form_Last5': 'Home_Form',
    'Rank': 'Home_Rank',
    'Streak': 'Home_Streak',
    'Goal_Diff_Cumul': 'Home_Goal_Diff',
    'Rolling_Goals_For': 'Home_Rolling_GF',
    'Rolling_Goals_Against': 'Home_Rolling_GA',
    'Points_Pace': 'Home_Points_Pace',
    'Match_Number': 'Home_Match_Number',
    'Points_Cumul': 'Home_Points_Cumul'
})

# Renommage colonnes Away
df_away = df_away.rename(columns={
    'Team': 'AwayTeam',
    'Opponent': 'HomeTeam',
    'Goals_For': 'FTAG',
    'Goals_Against': 'FTHG',
    'Form_Last5': 'Away_Form',
    'Rank': 'Away_Rank',
    'Streak': 'Away_Streak',
    'Goal_Diff_Cumul': 'Away_Goal_Diff',
    'Rolling_Goals_For': 'Away_Rolling_GF',
    'Rolling_Goals_Against': 'Away_Rolling_GA',
    'Points_Pace': 'Away_Points_Pace',
    'Match_Number': 'Away_Match_Number',
    'Points_Cumul': 'Away_Points_Cumul'
})

# Colonnes à garder
home_cols = ['Date', 'Season', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'Result', 'Referee',
             'Home_Form', 'Home_Rank', 'Home_Streak', 'Home_Goal_Diff', 
             'Home_Rolling_GF', 'Home_Rolling_GA', 'Home_Points_Pace', 'Home_Match_Number']

away_cols = ['Date', 'Season', 'HomeTeam', 'AwayTeam',
             'Away_Form', 'Away_Rank', 'Away_Streak', 'Away_Goal_Diff',
             'Away_Rolling_GF', 'Away_Rolling_GA', 'Away_Points_Pace', 'Away_Match_Number']

df_home = df_home[home_cols]
df_away = df_away[away_cols]

# Fusion sur (Date, Season, HomeTeam, AwayTeam)
df_final = df_home.merge(
    df_away,
    on=['Date', 'Season', 'HomeTeam', 'AwayTeam'],
    how='inner'
)

# Renommage Result → FTR
df_final = df_final.rename(columns={'Result': 'FTR'})

print(f"✓ Format match-complete créé : {df_final.shape}")
print(f"\nColonnes finales ({len(df_final.columns)}) :")
print(df_final.columns.tolist())
print(f"\nAperçu (5 premiers matchs) :")
print(df_final.head())


✓ Format match-complete créé : (12614, 24)

Colonnes finales (24) :
['Date', 'Season', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'FTR', 'Referee', 'Home_Form', 'Home_Rank', 'Home_Streak', 'Home_Goal_Diff', 'Home_Rolling_GF', 'Home_Rolling_GA', 'Home_Points_Pace', 'Home_Match_Number', 'Away_Form', 'Away_Rank', 'Away_Streak', 'Away_Goal_Diff', 'Away_Rolling_GF', 'Away_Rolling_GA', 'Away_Points_Pace', 'Away_Match_Number']

Aperçu (5 premiers matchs) :
        Date  Season HomeTeam     AwayTeam  FTHG  FTAG FTR        Referee  \
0 2000-08-21       1  Arsenal    Liverpool   2.0   0.0   H    Graham Poll   
1 2000-08-26       1  Arsenal     Charlton   5.0   3.0   H    Steve Lodge   
2 2000-09-16       1  Arsenal     Coventry   2.0   1.0   H      Mike Dean   
3 2000-10-01       1  Arsenal   Man United   1.0   0.0   H  Graham Barber   
4 2000-10-14       1  Arsenal  Aston Villa   1.0   0.0   H     Rob Harris   

   Home_Form  Home_Rank  ...  Home_Points_Pace  Home_Match_Number  Away_Form  \
0     

In [17]:
print("="*70)
print("STATISTIQUES DESCRIPTIVES DES FEATURES")
print("="*70)

# Nombre de NaN par feature
print("\n1. Valeurs manquantes par feature :")
nan_counts = df_final[['Home_Form', 'Home_Rank', 'Home_Streak', 'Home_Goal_Diff',
                        'Away_Form', 'Away_Rank', 'Away_Streak', 'Away_Goal_Diff']].isna().sum()
print(nan_counts)

# Distribution des features principales
print("\n2. Distribution Home_Form (0-15 points attendus) :")
print(df_final['Home_Form'].describe())

print("\n3. Distribution Home_Rank (1-22 attendu) :")
print(df_final['Home_Rank'].describe())

print("\n4. Distribution Home_Streak (-14 à +18 selon EDA) :")
print(df_final['Home_Streak'].describe())

print("\n5. Distribution Away_Form :")
print(df_final['Away_Form'].describe())

# Vérification cohérence avec EDA
print("\n6. Distribution FTR (résultats) :")
print(df_final['FTR'].value_counts(normalize=True).sort_index())
print("   → Attendu : ~45.7% H, ~25.5% D, ~28.9% A (cf FINDINGS)")

# Proportion de données utilisables (sans NaN)
complete_cases = df_final.dropna(subset=['Home_Form', 'Away_Form']).shape[0]
print(f"\n7. Matchs avec features complètes (sans NaN) : {complete_cases}/{len(df_final)} ({complete_cases/len(df_final)*100:.1f}%)")


STATISTIQUES DESCRIPTIVES DES FEATURES

1. Valeurs manquantes par feature :
Home_Form         332
Home_Rank           0
Home_Streak         0
Home_Goal_Diff    332
Away_Form         332
Away_Rank           0
Away_Streak         0
Away_Goal_Diff    332
dtype: int64

2. Distribution Home_Form (0-15 points attendus) :
count    12282.000000
mean         6.367448
std          3.454388
min          0.000000
25%          4.000000
50%          6.000000
75%          9.000000
max         15.000000
Name: Home_Form, dtype: float64

3. Distribution Home_Rank (1-22 attendu) :
count    12614.000000
mean         6.304424
std          4.559751
min          1.000000
25%          2.000000
50%          5.000000
75%         10.000000
max         22.000000
Name: Home_Rank, dtype: float64

4. Distribution Home_Streak (-14 à +18 selon EDA) :
count    12614.000000
mean        -0.103298
std          1.875504
min        -14.000000
25%         -1.000000
50%          0.000000
75%          1.000000
max         17.0

---

## 5. Nettoyage final et export

**Décisions sur les valeurs manquantes** :

1. **332 matchs avec NaN** (2.7%) = premiers matchs de saison (pas d'historique)
2. **Options** :
   - A) Supprimer ces lignes (12 279 matchs utilisables)
   - B) Garder avec NaN (modèles ML peuvent gérer)
   - C) Imputation (ex: Form=0, Goal_Diff=0, Points_Pace=médiane saison)

**Décision** : Option A (suppression) pour v1, car :
- Perte minime (2.7%)
- Évite biais d'imputation
- Premiers matchs peu prédictifs de toute façon (pas d'historique)


In [18]:
# Suppression des matchs avec features manquantes
df_final_clean = df_final.dropna(subset=['Home_Form', 'Away_Form']).reset_index(drop=True)

print(f"Dataset avant nettoyage : {len(df_final)} matchs")
print(f"Dataset après nettoyage : {len(df_final_clean)} matchs")
print(f"Matchs supprimés : {len(df_final) - len(df_final_clean)} ({(len(df_final) - len(df_final_clean))/len(df_final)*100:.2f}%)")

# Vérification absence de NaN
print(f"\nVérification NaN restants :")
print(df_final_clean.isna().sum().sum(), "NaN trouvés")

# Tri chronologique
df_final_clean = df_final_clean.sort_values(['Season', 'Date']).reset_index(drop=True)

# Export
output_path = DATA_PROCESSED / 'pl_features_v1.csv'
df_final_clean.to_csv(output_path, index=False)

print(f"\n{'='*70}")
print(f"✓ EXPORT RÉUSSI")
print(f"{'='*70}")
print(f"Fichier : {output_path}")
print(f"Taille : {output_path.stat().st_size / 1024:.1f} Ko")
print(f"Lignes : {len(df_final_clean)}")
print(f"Colonnes : {len(df_final_clean.columns)}")
print(f"Période : {df_final_clean['Date'].min()} → {df_final_clean['Date'].max()}")
print(f"Saisons : {df_final_clean['Season'].nunique()}")

print(f"\nColonnes exportées :")
for i, col in enumerate(df_final_clean.columns, 1):
    print(f"  {i:2d}. {col}")


Dataset avant nettoyage : 12614 matchs
Dataset après nettoyage : 12279 matchs
Matchs supprimés : 335 (2.66%)

Vérification NaN restants :
2752 NaN trouvés

✓ EXPORT RÉUSSI
Fichier : /workspace/data/processed/pl_features_v1.csv
Taille : 1607.7 Ko
Lignes : 12279
Colonnes : 24
Période : 1993-08-16 00:00:00 → 2026-05-24 00:00:00
Saisons : 33

Colonnes exportées :
   1. Date
   2. Season
   3. HomeTeam
   4. AwayTeam
   5. FTHG
   6. FTAG
   7. FTR
   8. Referee
   9. Home_Form
  10. Home_Rank
  11. Home_Streak
  12. Home_Goal_Diff
  13. Home_Rolling_GF
  14. Home_Rolling_GA
  15. Home_Points_Pace
  16. Home_Match_Number
  17. Away_Form
  18. Away_Rank
  19. Away_Streak
  20. Away_Goal_Diff
  21. Away_Rolling_GF
  22. Away_Rolling_GA
  23. Away_Points_Pace
  24. Away_Match_Number


In [19]:
print("Diagnostic des NaN restants :")
print("="*70)

nan_by_column = df_final_clean.isna().sum()
nan_by_column = nan_by_column[nan_by_column > 0].sort_values(ascending=False)

print(f"\nNaN par colonne :")
print(nan_by_column)

print(f"\nExemple de lignes avec NaN dans Points_Pace :")
sample_nan = df_final_clean[df_final_clean['Home_Points_Pace'].isna()].head(5)
print(sample_nan[['Date', 'HomeTeam', 'AwayTeam', 'Home_Match_Number', 
                   'Home_Points_Pace', 'Away_Points_Pace', 'FTR']])

print(f"\n→ Ces NaN sont attendus pour Match_Number=2 (division par 1, puis projection instable)")
print(f"→ Options : 1) Imputer avec 0 ou médiane, 2) Laisser (XGBoost gère les NaN), 3) Supprimer feature")


Diagnostic des NaN restants :

NaN par colonne :
Referee    2752
dtype: int64

Exemple de lignes avec NaN dans Points_Pace :
Empty DataFrame
Columns: [Date, HomeTeam, AwayTeam, Home_Match_Number, Home_Points_Pace, Away_Points_Pace, FTR]
Index: []

→ Ces NaN sont attendus pour Match_Number=2 (division par 1, puis projection instable)
→ Options : 1) Imputer avec 0 ou médiane, 2) Laisser (XGBoost gère les NaN), 3) Supprimer feature


---

## 6. Résumé et prochaines étapes

### ✅ Réalisations

**Dataset créé** : `pl_features_v1.csv`
- **12 279 matchs** (97.3% des données originales)
- **33 saisons** (1993-2026)
- **24 colonnes** : métadonnées + 8 features × 2 équipes (Home/Away)

**Features validées statistiquement** (notebook 04) :
1. ✅ **Form_Last5** (η² = 0.0160, +57% impact)
2. ✅ **Rank** (η² = 0.0300, +81% impact, meilleure taille d'effet)
3. ✅ **Streak** (η² = 0.0083, +97% aux extrêmes)
4. ✅ **IsHome** (implicite via séparation Home/Away)

**Features complémentaires** :
5. ✅ **Goal_Diff_Cumul** (proxy force brute)
6. ✅ **Rolling_Goals_For/Against** (tendances offensives/défensives)
7. ✅ **Points_Pace** (projection 38 matchs)
8. ✅ **Match_Number** (position saison)

**Garantie anti-leakage** : `.shift(1)` appliqué systématiquement → toutes les features calculées avec infos **avant** le match.

### 📊 Statistiques finales

- **Distribution FTR** : 45.6% H / 25.5% D / 28.9% A (conforme FINDINGS)
- **Valeurs manquantes** : 0 pour features numériques, 2752 pour `Referee` (non utilisé)
- **Perte de données** : 335 matchs (2.7%, premiers matchs saison sans historique)

### ⏭️ Prochaines étapes

1. **Notebook 05bis** (optionnel, après scrapers) : Ajout xG + valeurs marchandes
2. **Notebook 06** : Modélisation (baseline → modèles complexes)
3. **Notebook 07** : Prédiction 2026/27 + carte geopandas

---

## 🎯 Dataset prêt pour la modélisation !


In [20]:
# Chargement du fichier exporté pour vérification
df_verify = pd.read_csv(DATA_PROCESSED / 'pl_features_v1.csv')

print("="*70)
print("VÉRIFICATION FINALE DU FICHIER EXPORTÉ")
print("="*70)

print(f"\n✓ Fichier chargé avec succès")
print(f"  Shape : {df_verify.shape}")
print(f"  Taille : {(DATA_PROCESSED / 'pl_features_v1.csv').stat().st_size / 1024:.1f} Ko")

print(f"\n✓ Aperçu des 3 premiers matchs :")
print(df_verify.head(3))

print(f"\n✓ Aperçu des 3 derniers matchs (saison 2025/26) :")
print(df_verify.tail(3))

print(f"\n✓ Statistiques features Home vs Away (symétrie attendue) :")
print(f"  Home_Form : mean={df_verify['Home_Form'].mean():.2f}, std={df_verify['Home_Form'].std():.2f}")
print(f"  Away_Form : mean={df_verify['Away_Form'].mean():.2f}, std={df_verify['Away_Form'].std():.2f}")
print(f"  Home_Rank : mean={df_verify['Home_Rank'].mean():.2f}, std={df_verify['Home_Rank'].std():.2f}")
print(f"  Away_Rank : mean={df_verify['Away_Rank'].mean():.2f}, std={df_verify['Away_Rank'].std():.2f}")

print(f"\n{'='*70}")
print(f"✅ NOTEBOOK 05 TERMINÉ — Dataset v1 prêt pour modélisation")
print(f"{'='*70}")


VÉRIFICATION FINALE DU FICHIER EXPORTÉ

✓ Fichier chargé avec succès
  Shape : (12279, 24)
  Taille : 1607.7 Ko

✓ Aperçu des 3 premiers matchs :
         Date  Season  HomeTeam    AwayTeam  FTHG  FTAG FTR      Referee  \
0  2000-08-21       1   Arsenal   Liverpool   2.0   0.0   H  Graham Poll   
1  2000-08-22       1  Bradford     Chelsea   2.0   0.0   H  Mark Halsey   
2  2000-08-22       1   Ipswich  Man United   1.0   1.0   D  Jeff Winter   

   Home_Form  Home_Rank  ...  Home_Points_Pace  Home_Match_Number  Away_Form  \
0        0.0        2.0  ...               0.0                  2        3.0   
1        0.0        5.0  ...               0.0                  2        3.0   
2        0.0        6.0  ...               0.0                  2        3.0   

   Away_Rank  Away_Streak  Away_Goal_Diff  Away_Rolling_GF  Away_Rolling_GA  \
0        1.0            1             1.0              1.0              0.0   
1        1.0            1             2.0              4.0            

---

## 📝 Notes méthodologiques

### Choix de conception

**1. Format "match-complete" vs "expanded"**
- ✅ Choisi : 1 ligne = 1 match (12.2K lignes)
- Raison : Format standard ML football, features des 2 équipes côte à côté
- Alternative : 1 ligne = 1 équipe (25K lignes) → redondance, moins adapté à prédiction H/D/A

**2. Gestion data leakage**
- ✅ `.shift(1)` systématique sur tous les rolling/cumsum
- Vérification : Match 1 de chaque saison = NaN ou 0 (pas d'historique)
- Conséquence : 335 matchs supprimés (2.7%), acceptable

**3. Fenêtres temporelles**
- ✅ Rolling 5 matchs (Form, Goals) : compromis récence/stabilité
- Alternative testée dans EDA : 3 matchs (trop volatile), 10 matchs (trop lissé)

**4. Valeurs manquantes**
- ✅ Suppression des 335 matchs sans historique (premiers de saison)
- Alternative : Imputation (Form=0, Goal_Diff=0) → risque biais
- NaN `Referee` (2752) : conservés, colonne non utilisée en modélisation

### Limites de la v1

- ❌ Pas de xG (données FBref, disponibles ~2017+)
- ❌ Pas de valeurs marchandes (Transfermarkt, ~2010+)
- ❌ Pas de lineups (API-Football temps réel uniquement)
- ❌ Pas d'interactions entre features (reporté au notebook 06)

→ Ces enrichissements seront ajoutés dans **notebook 05bis** après développement des scrapers.

### Points d'attention pour notebook 06

1. **Train/test split temporel** : ne pas mélanger saisons (risque leakage)
2. **Validation croisée adaptée** : TimeSeriesSplit, pas KFold standard
3. **Encodage IsHome** : déjà implicite (colonnes Home/Away séparées)
4. **Features à normaliser** : Points_Pace (échelle 0-114), autres déjà homogènes (0-15, 1-22)
5. **Target déséquilibrée** : 45.6% H, 25.5% D, 28.9% A → considérer class_weight
